In [ ]:
import json
import os
import hashlib
from pathlib import Path


# ─────────────────────────────────────────────────────────────────────────────
# HOW THE CACHE WORKS
# ─────────────────────────────────────────────────────────────────────────────
# Each chunk's text is hashed (SHA-256) → used as the cache key.
# Cache is stored as a single JSON file: { "hash": [embedding vector] }
# On rerun: if hash exists in cache → skip API call, load from disk.
#           if hash is new          → call OpenAI, store result in cache.
# This means:
#   - Unchanged chunks are never re-embedded (saves cost + time)
#   - New chunks added to existing PDFs are embedded and merged in
#   - Editing a chunk's text creates a new hash → re-embedded automatically


CACHE_PATH = "./embedding_cache.json"


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load cache from disk
# ─────────────────────────────────────────────────────────────────────────────

def load_cache(cache_path: str = CACHE_PATH) -> dict:
    """
    Loads existing embedding cache from disk.
    Returns an empty dict if no cache file exists yet.
    """
    if Path(cache_path).exists():
        with open(cache_path, "r") as f:
            cache = json.load(f)
        print(f"✅ Loaded embedding cache: {len(cache)} embeddings from '{cache_path}'")
        return cache
    else:
        print(f"  No cache found at '{cache_path}' — starting fresh")
        return {}


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Save cache to disk
# ─────────────────────────────────────────────────────────────────────────────

def save_cache(cache: dict, cache_path: str = CACHE_PATH) -> None:
    """
    Saves the current embedding cache to disk.
    Always call this after adding new embeddings.
    """
    with open(cache_path, "w") as f:
        json.dump(cache, f)
    print(f"✅ Cache saved: {len(cache)} embeddings → '{cache_path}'")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Hash a text (cache key)
# ─────────────────────────────────────────────────────────────────────────────

def hash_text(text: str) -> str:
    """
    Creates a unique SHA-256 hash for a chunk's text.
    Used as the cache lookup key.
    If text changes even slightly → different hash → re-embedded.
    """
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — Get embeddings with caching (replaces raw get_embeddings calls)
# ─────────────────────────────────────────────────────────────────────────────

def get_embeddings_cached(
    texts: list[str],
    cache: dict,
    model: str = "text-embedding-3-small",
    batch_size: int = 500,
) -> tuple[list[list[float]], dict]:
    """
    Returns embeddings for all texts — from cache where possible,
    from OpenAI API only for new texts.

    Args:
        texts:      list of strings to embed
        cache:      the loaded cache dict (from load_cache)
        model:      OpenAI embedding model
        batch_size: how many new texts to send per API call

    Returns:
        embeddings: list of vectors in same order as input texts
        cache:      updated cache dict (call save_cache after this)
    """
    from rag_pipeline import get_embeddings  # your existing function

    # ── Split into cached vs new ──────────────────────────────────────────────
    hashes       = [hash_text(t) for t in texts]
    cached_hits  = [h for h in hashes if h in cache]
    new_texts    = [(i, texts[i]) for i, h in enumerate(hashes) if h not in cache]

    print(f"  Cache hits  : {len(cached_hits)}/{len(texts)} (skipping API call)")
    print(f"  New texts   : {len(new_texts)} (calling OpenAI)")

    # ── Embed only new texts ──────────────────────────────────────────────────
    if new_texts:
        indices, raw_texts = zip(*new_texts)
        new_embeddings = get_embeddings(list(raw_texts), model=model, batch_size=batch_size)

        # Store new results in cache
        for text, embedding in zip(raw_texts, new_embeddings):
            cache[hash_text(text)] = embedding

    # ── Assemble final list in original order ─────────────────────────────────
    embeddings = [cache[h] for h in hashes]

    return embeddings, cache


# ─────────────────────────────────────────────────────────────────────────────
# STEP 5 — Attach embeddings to chunks (convenience wrapper)
# ─────────────────────────────────────────────────────────────────────────────

def embed_chunks(
    chunks: list[dict],
    cache_path: str = CACHE_PATH,
    model: str = "text-embedding-3-small",
) -> list[dict]:
    """
    Embeds the contextualized_text of each chunk, using cache where possible.
    Attaches the embedding directly onto each chunk dict as chunk["embedding"].
    Saves the updated cache to disk automatically.

    This is the function to call between add_contextual_retrieval()
    and add_chunks_to_collection() in your pipeline.
    """
    cache = load_cache(cache_path)

    texts = [c["contextualized_text"] for c in chunks]
    embeddings, cache = get_embeddings_cached(texts, cache, model=model)

    # Attach embedding to each chunk
    for chunk, embedding in zip(chunks, embeddings):
        chunk["embedding"] = embedding

    # Always save after new embeddings are added
    save_cache(cache, cache_path)

    return chunks


# ─────────────────────────────────────────────────────────────────────────────
# STEP 6 — Cache utilities
# ─────────────────────────────────────────────────────────────────────────────

def cache_stats(cache_path: str = CACHE_PATH) -> None:
    """Shows a summary of the current cache."""
    if not Path(cache_path).exists():
        print("No cache file found")
        return

    cache = load_cache(cache_path)
    size_mb = os.path.getsize(cache_path) / (1024 * 1024)

    print(f"\n📊 Embedding Cache Stats")
    print(f"   Path            : {cache_path}")
    print(f"   Total embeddings: {len(cache)}")
    print(f"   File size       : {size_mb:.2f} MB")

    if cache:
        sample_vector = next(iter(cache.values()))
        print(f"   Vector dimensions: {len(sample_vector)}")


def clear_cache(cache_path: str = CACHE_PATH) -> None:
    """
    Deletes the cache file entirely.
    Use when switching embedding models — old vectors won't be
    compatible with a new model and must be recomputed.
    """
    if Path(cache_path).exists():
        os.remove(cache_path)
        print(f"🗑️  Cache cleared at '{cache_path}'")
    else:
        print("No cache file to clear")